<a href="https://colab.research.google.com/github/chaeyoungwon/mask-aware-inpainting/blob/yeonholee/gated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/mask-inpainting-checkpoints'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os

%cd /content

repo_name = 'mask-aware-inpainting'
repo_path = os.path.join('/content', repo_name)

# Check if the directory already exists. If it does, remove it to ensure a clean clone.
# This addresses the "destination path 'mask-aware-inpainting' already exists" error
# and implicitly handles the "not a git repository" if the existing directory is corrupted.
if os.path.exists(repo_path):
    print(f"Removing existing '{repo_name}' directory to perform a clean clone.")
    !rm -rf {repo_name}

# Perform the clone
!git clone https://github.com/chaeyoungwon/mask-aware-inpainting.git

# Change into the cloned directory
%cd {repo_name}

# Now that we are inside the repository, execute git pull
!git pull origin main

/content
Removing existing 'mask-aware-inpainting' directory to perform a clean clone.
Cloning into 'mask-aware-inpainting'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 196 (delta 12), reused 14 (delta 7), pack-reused 170 (from 1)
Receiving objects: 100% (196/196), 520.13 KiB | 2.42 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/mask-aware-inpainting
From https://github.com/chaeyoungwon/mask-aware-inpainting
 * branch            main       -> FETCH_HEAD
Already up to date.


## 데이터셋 구성

In [18]:
# kaggle.json 업로드
from google.colab import files
files.upload()  # kaggle.json 선택

# 설치 및 인증
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [19]:
!kaggle datasets download -d jessicali9530/celeba-dataset -p ./mask-aware-inpainting/data/
!cd mask-aware-inpainting/data && unzip celeba-dataset.zip -d celeba_temp

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197604.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197605.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197606.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197607.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197608.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197609.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197610.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197611.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197612.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197613.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197614.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197615.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197616.jpg  
  inflating: celeba_temp/img

In [23]:
import os

# 이미지를 올바른 위치로 이동
src = '/content/mask-aware-inpainting/mask-aware-inpainting/data/celeba_temp/img_align_celeba/img_align_celeba'
dst = '/content/mask-aware-inpainting/data/celeba/img_align_celeba'

os.makedirs('/content/mask-aware-inpainting/data/celeba', exist_ok=True)
os.rename(src, dst)
print("완료:", len(os.listdir(dst)), "개")

base = '/content/mask-aware-inpainting/data/celeba'
filenames = sorted(os.listdir(f'{base}/img_align_celeba'))
n = len(filenames)

# list_eval_partition.txt
with open(f'{base}/list_eval_partition.txt', 'w') as f:
    for i, fname in enumerate(filenames):
        split = 0 if i < 162770 else (1 if i < 182637 else 2)
        f.write(f'{fname} {split}\n')

# identity
with open(f'{base}/identity_CelebA.txt', 'w') as f:
    for fname in filenames:
        f.write(f'{fname} 1\n')

# attr (더미)
attrs = ['5_o_Clock_Shadow','Arched_Eyebrows','Attractive','Bags_Under_Eyes','Bald','Bangs','Big_Lips','Big_Nose','Black_Hair','Blond_Hair','Blurry','Brown_Hair','Bushy_Eyebrows','Chubby','Double_Chin','Eyeglasses','Goatee','Gray_Hair','Heavy_Makeup','High_Cheekbones','Male','Mouth_Slightly_Open','Mustache','Narrow_Eyes','No_Beard','Oval_Face','Pale_Skin','Pointy_Nose','Receding_Hairline','Rosy_Cheeks','Sideburns','Smiling','Straight_Hair','Wavy_Hair','Wearing_Earrings','Wearing_Hat','Wearing_Lipstick','Wearing_Necklace','Wearing_Necktie','Young']
with open(f'{base}/list_attr_celeba.txt', 'w') as f:
    f.write(f'{n}\n{" ".join(attrs)}\n')
    for fname in filenames:
        f.write(fname + ' ' + ' '.join(['-1']*40) + '\n')

# bbox / landmarks (더미)
with open(f'{base}/list_bbox_celeba.txt', 'w') as f:
    f.write('image_id x_1 y_1 width height\n')
    for fname in filenames:
        f.write(f'{fname} 0 0 128 128\n')

with open(f'{base}/list_landmarks_align_celeba.txt', 'w') as f:
    f.write(f'{n}\nlefteye_x lefteye_y righteye_x righteye_y nose_x nose_y leftmouth_x leftmouth_y rightmouth_x rightmouth_y\n')
    for fname in filenames:
        f.write(f'{fname} 32 32 96 32 64 64 32 96 96 96\n')

print("메타파일 완료")
print(os.listdir(base))

완료: 202599 개
메타파일 완료
['list_eval_partition.txt', 'identity_CelebA.txt', 'list_bbox_celeba.txt', 'img_align_celeba', 'list_attr_celeba.txt', 'list_landmarks_align_celeba.txt']


In [24]:
# download=False 실제로 저장됐는지 확인
!grep "download" /content/mask-aware-inpainting/datasets/celeba_dataset.py

# 메타데이터 파일 있는지 확인
import os
celeba_dir = '/content/mask-aware-inpainting/data/celeba'
if os.path.exists(celeba_dir):
    print("celeba dir contents:", os.listdir(celeba_dir))
else:
    print("celeba dir does NOT exist!")

    root        : dataset root (CelebA will be downloaded here if needed)
            celeba = CelebA(root=root, split=split, transform=transform, download=False)
celeba dir contents: ['list_eval_partition.txt', 'identity_CelebA.txt', 'list_bbox_celeba.txt', 'img_align_celeba', 'list_attr_celeba.txt', 'list_landmarks_align_celeba.txt']


In [25]:
# 이미지가 어디 있는지 확인
import glob

paths_to_check = [
    '/content/mask-aware-inpainting/data/celeba/img_align_celeba',
    '/content/mask-aware-inpainting/data/celeba/img_align_celeba/img_align_celeba',
    '/content/mask-aware-inpainting/mask-aware-inpainting/data/celeba/img_align_celeba',
]
for p in paths_to_check:
    if os.path.exists(p):
        files = os.listdir(p)
        print(f"✅ {p}: {len(files)} files  (e.g. {files[:3]})")
    else:
        print(f"❌ {p}: not found")

✅ /content/mask-aware-inpainting/data/celeba/img_align_celeba: 202599 files  (e.g. ['139945.jpg', '089570.jpg', '034660.jpg'])
❌ /content/mask-aware-inpainting/data/celeba/img_align_celeba/img_align_celeba: not found
❌ /content/mask-aware-inpainting/mask-aware-inpainting/data/celeba/img_align_celeba: not found


In [26]:
import sys, importlib
sys.path.insert(0, '/content/mask-aware-inpainting')
%cd /content/mask-aware-inpainting

import datasets.celeba_dataset as celeba_dataset
importlib.reload(celeba_dataset)

CelebAInpaintingDataset = celeba_dataset.CelebAInpaintingDataset

/content/mask-aware-inpainting


In [27]:
from datasets.celeba_dataset import CelebAInpaintingDataset

In [28]:
from config import Config
import random, os
import numpy as np
import torch

random.seed(42); np.random.seed(42); torch.manual_seed(42)
cfg = Config()

dataset = CelebAInpaintingDataset(
    root=cfg.data_root,
    split='test',
    image_size=cfg.image_size,
    max_samples=500
)

print(f"Dataset size: {len(dataset)}")

Dataset size: 500


In [29]:
!python3 prepare_fixed_testset.py

Saving 500 samples to ./fixed_testset/  (seed=42) ...
  100/500
  200/500
  300/500
  400/500
  500/500

Saved 500 samples → ./fixed_testset/
Hole ratio — mean: 0.499 | min: 0.018 | max: 0.874
  small  (< 0.3)      : 108 samples
  medium (0.3 – 0.5)  : 125 samples
  large  (0.5 – 0.7)  : 158 samples


In [30]:
# conv_type을 'gated'으로 설정
import re
with open('config.py', 'r') as f:
    _cfg = f.read()
_cfg = re.sub(r"conv_type\s*=\s*'[^']*'", "conv_type = 'gated'", _cfg)
with open('config.py', 'w') as f:
    f.write(_cfg)
print("conv_type set to 'gated'")


conv_type set to 'gated'


## 학습

In [ ]:
!python3 train.py

device: cuda  |  conv_type: gated  |  seed: 42
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:04<00:00, 131MB/s]
Trainable parameters: 20,291,139
  [1/50] step    0 | loss 2.3180 | valid 0.3352 | hole 0.2643 | perc 7.9362
  [1/50] step   20 | loss 1.3032 | valid 0.0861 | hole 0.1530 | perc 5.9865
  [1/50] step   40 | loss 1.2039 | valid 0.0538 | hole 0.1445 | perc 5.6613
  [1/50] step   60 | loss 1.1257 | valid 0.0473 | hole 0.1333 | perc 5.5659
  [1/50] step   80 | loss 0.9873 | valid 0.0414 | hole 0.1133 | perc 5.3248
  [1/50] step  100 | loss 0.8296 | valid 0.0440 | hole 0.0928 | perc 4.5732
  [1/50] step  120 | loss 0.9680 | valid 0.0338 | hole 0.1127 | perc 5.1570
  [1/50] step  140 | loss 0.9659 | valid 0.0295 | hole 0.1132 | perc 5.1386
  [1/50] step  160 | loss 1.0185 | valid 0.0539 | hole 0.1200 | perc 4.8908
  [1/50] step  180 | loss 0.8838 | valid 0.0248 | hole 0.1039 | perc 4.

In [ ]:
!python evaluate.py checkpoints/gated/best_model.pth --conv_type gated --fixed_testset ./fixed_testset


## 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('checkpoints/gated/training_history_gated.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Gated CAE', fontsize=14)

ax1.plot(df['epoch'], df['val_psnr'], color='steelblue')
ax1.set_title('Epoch별 Val PSNR')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('PSNR (dB)'); ax1.grid(True)

ax2.plot(df['epoch'], df['val_ssim'], color='goldenrod')
ax2.set_title('Epoch별 Val SSIM')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('SSIM'); ax2.grid(True)

plt.tight_layout()
plt.show()
print(f'최종 PSNR: {df["val_psnr"].iloc[-1]:.2f} dB  |  최종 SSIM: {df["val_ssim"].iloc[-1]:.4f}')
